# BRONZE LAYER

# Various Imports and Python Environment Configuration.

In [1]:
import os
import sys
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark import StorageLevel
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

# Done to force the Spark driver to use the same Python Interpreter as the kernel (otherwise problems regarding the workers will rise in my case).

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [3]:
print("Notebook Python:", sys.executable)
print("Spark Python   :", os.environ["PYSPARK_PYTHON"])

Notebook Python: C:\lufthansa-de-exercise\.venv\Scripts\python.exe
Spark Python   : C:\lufthansa-de-exercise\.venv\Scripts\python.exe


# Check the project root and other directories for this project. Create a bronze directory if needed.

In [4]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" 
BRONZE_DIR = PROJECT_ROOT / "delta" / "bronze"

BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw CSVs :", RAW_DIR)
print("Bronze :", BRONZE_DIR)

Project root: C:\lufthansa-de-exercise
Raw CSVs : C:\lufthansa-de-exercise\data
Bronze : C:\lufthansa-de-exercise\delta\bronze


# Create the spark session in combination with delta lake.

In [5]:
builder = (
    SparkSession.builder
    .appName("bronze-ingestion")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Declare the available tables in the project like a nested dictionary. Contains the table name, csv filename and timestamp column for partioning.     (4 tables didnt have timestamp columns so they wont be partioned.)

In [6]:
TABLES = {
    "orders": {
        "csv_file": "olist_orders_dataset.csv",
        "timestamp_col": "order_purchase_timestamp",
    },
    "order_items": {
        "csv_file": "olist_order_items_dataset.csv",
        "timestamp_col": "shipping_limit_date",
    },
    "order_reviews": {
        "csv_file": "olist_order_reviews_dataset.csv",
        "timestamp_col": "review_creation_date",
    },
    "order_payments": {
        "csv_file": "olist_order_payments_dataset.csv",
        "timestamp_col": None,
    },
    "customers": {
        "csv_file": "olist_customers_dataset.csv",
        "timestamp_col": None,
    },
    "products": {
        "csv_file": "olist_products_dataset.csv",
        "timestamp_col": None,
    },
    "sellers": {
        "csv_file": "olist_sellers_dataset.csv",
        "timestamp_col": None,
    },
}

# Simple display of the nested dictionary above.

In [7]:
for table_name, config in TABLES.items():
    timestamp_col = config["timestamp_col"] or "(no timestamp)"

    print(
        f"{table_name:<16} "
        f"{config['csv_file']:<36} "
        f"{timestamp_col}"
    )

orders           olist_orders_dataset.csv             order_purchase_timestamp
order_items      olist_order_items_dataset.csv        shipping_limit_date
order_reviews    olist_order_reviews_dataset.csv      review_creation_date
order_payments   olist_order_payments_dataset.csv     (no timestamp)
customers        olist_customers_dataset.csv          (no timestamp)
products         olist_products_dataset.csv           (no timestamp)
sellers          olist_sellers_dataset.csv            (no timestamp)


# Declare the bronze ingestion function. Used to read all the csv files, creates the partitions when needed and writes the result into the bronze delta tables.

In [8]:
def ingest_to_bronze(
     # It uses 3 arguments, table name, csv file and timestamp column with a default value None. It returns a dictionary.
    table_name: str,
    csv_file: str,
    timestamp_col: str | None = None,
) -> dict:

    # assign the source and destination paths.
    source_path = RAW_DIR / csv_file
    output_path = BRONZE_DIR / table_name

    print(f"\nProcessing table: {table_name}")
    print(f"Source            : {source_path}")
    print(f"Destination       : {output_path}")
    
    # loads the csv file into a datafrome.
    # Last 3 options are used for any double quotes, escaped ones and multilines that can happen especially in the order reviews table.
    df = spark.read.csv(
        str(source_path),
        header=True,
        inferSchema=True,
        quote='"',
        escape='"',
        multiLine=True,
    )

    # create the partition by year, month, date
    partition_cols: list[str] = []

    #check tables that dont have any timestamps for partition and skip this block
    if timestamp_col is not None:
        if timestamp_col not in df.columns:
            raise ValueError(
                f"Timestamp column '{timestamp_col}' was not found "
                f"in table '{table_name}'. "
                f"Available columns: {df.columns}"
            )

        parsed_timestamp = F.to_timestamp(
                F.col(timestamp_col)
        )
    
        df = (
            df.withColumn(
                "year",
                F.year(parsed_timestamp),
            )
            .withColumn(
                "month",
                F.month(parsed_timestamp),
            )
            .withColumn(
                "day",
                F.dayofmonth(parsed_timestamp),
            )
        )
    
        partition_cols = ["year", "month", "day"]

    # starting the write process to bronze layer
    source_count = df.count()

    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )

    # process of saving the tables by the partitions into the folder
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    
    writer.save(str(output_path))

    # check the count for each table
    print(f"Rows written      : {source_count:,}")
    print(f"Partitions       : {partition_cols if partition_cols else 'none'}")

    # return the metadata in form of a dictionary for each table
    return {
        "table_name": table_name,
        "source_path": str(source_path),
        "output_path": str(output_path),
        "source_count": source_count,
        "partition_cols": partition_cols,
    }

# Running the bronze ingestion for every table.

In [9]:
ingestion_results = {}

for table_name, config in TABLES.items():
    result = ingest_to_bronze(
        table_name=table_name,
        csv_file=config["csv_file"],
        timestamp_col=config["timestamp_col"],
    )
    # storing the results in a dictionary
    ingestion_results[table_name] = result



Processing table: orders
Source            : C:\lufthansa-de-exercise\data\olist_orders_dataset.csv
Destination       : C:\lufthansa-de-exercise\delta\bronze\orders
Rows written      : 99,441
Partitions       : ['year', 'month', 'day']

Processing table: order_items
Source            : C:\lufthansa-de-exercise\data\olist_order_items_dataset.csv
Destination       : C:\lufthansa-de-exercise\delta\bronze\order_items
Rows written      : 112,650
Partitions       : ['year', 'month', 'day']

Processing table: order_reviews
Source            : C:\lufthansa-de-exercise\data\olist_order_reviews_dataset.csv
Destination       : C:\lufthansa-de-exercise\delta\bronze\order_reviews
Rows written      : 99,224
Partitions       : ['year', 'month', 'day']

Processing table: order_payments
Source            : C:\lufthansa-de-exercise\data\olist_order_payments_dataset.csv
Destination       : C:\lufthansa-de-exercise\delta\bronze\order_payments
Rows written      : 103,886
Partitions       : none

Processin

# Simple checks in the bronze layer for the schema format and select statements for orders table.

In [10]:
orders_path = BRONZE_DIR / "orders"

orders = (
    spark.read
    .format("delta")
    .load(str(orders_path))
)

orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)



In [11]:
orders.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "year",
    "month",
    "day",
).show(
    5,
    truncate=False,
)

+--------------------------------+------------+------------------------+----+-----+---+
|order_id                        |order_status|order_purchase_timestamp|year|month|day|
+--------------------------------+------------+------------------------+----+-----+---+
|1b2d622f7ebb8e65f7de3e389f052540|delivered   |2016-10-06 11:05:43     |2016|10   |6  |
|6ece326e25f193d084e3dc092bbdd93b|delivered   |2016-10-06 20:15:20     |2016|10   |6  |
|323dad8c483c7f8b818a825d257f4aa0|delivered   |2016-10-06 20:05:04     |2016|10   |6  |
|2e32dea8a4d1ae5499a67674b387bc6a|delivered   |2016-10-06 02:53:39     |2016|10   |6  |
|dc91dbf7defabdf37c52672a989bad1b|delivered   |2016-10-06 12:22:07     |2016|10   |6  |
+--------------------------------+------------+------------------------+----+-----+---+
only showing top 5 rows



In [12]:
spark.stop()